# Fusion des CSV IIIF et résolution des URL de redirection ("ark" → URL physique)

Ce notebook :
1. Lit tous les CSV d'un dossier, en gérant un encodage **UTF-7** pour certains fichiers
   (par ex. `JJ96.csv`) et **UTF-8** pour les autres.
2. Détecte automatiquement le séparateur de colonnes (tabulation par défaut, d'après
   l'exemple fourni).
3. Concatène tous les fichiers dans un DataFrame unique en vérifiant la cohérence des colonnes.
4. Résout, pour chaque URL `urlImage` (de type `ark:/...`), l'**URL physique finale**
   après redirection HTTP — sans télécharger le contenu de l'image (requêtes `HEAD`,
   avec repli sur `GET` en streaming si `HEAD` n'est pas supporté).
5. Écrit un CSV final avec :
   - `urlImage_arkId` : l'URL d'origine (ark)
   - `urlImage` : l'URL physique résolue
   - toutes les autres colonnes inchangées

**À faire avant de lancer :** ajustez la cellule de configuration ci-dessous
(chemin du dossier d'entrée, noms des fichiers à lire en UTF-7, etc.).

## 1. Configuration

In [1]:
# =========================================================
# CONFIGURATION — à adapter avant de lancer le notebook
# =========================================================

# Dossier contenant les CSV en entrée (recherche non récursive par défaut)
INPUT_DIR = "../List-of-images"

# Motif de recherche des fichiers (modifier si vos CSV n'ont pas l'extension .csv)
GLOB_PATTERN = "*image_data.csv"

# Fichier de sortie
OUTPUT_FILE = "./images_merged_with_resolved_urls.csv"

# Encodage par défaut de tous les CSV
DEFAULT_ENCODING = "utf-8"

# Liste des noms de fichiers (ou motifs) à lire en UTF-7 plutôt qu'en UTF-8.
# Comparaison sur le nom de fichier exact (insensible à la casse) ou sur une
# sous-chaîne présente dans le nom de fichier.
UTF7_FILENAMES = ["JJ096-JJ099_image_data.csv"]  # ajustez selon vos vrais noms de fichiers

# Force un séparateur explicite pour certains fichiers (plutôt que la détection automatique).
# Clé = sous-chaîne du nom de fichier, valeur = séparateur à utiliser.
SEP_OVERRIDES = {
    "JJ113_image_data.csv": ";",
}


# Nombre de requêtes HTTP en parallèle pour la résolution des redirections
MAX_WORKERS = 8

# Timeout (secondes) par requête HTTP
REQUEST_TIMEOUT = 15

# Nombre de tentatives en cas d'échec réseau
MAX_RETRIES = 5

# User-Agent poli pour les requêtes
USER_AGENT = "IRHT-redirect-resolver/1.0 (research use; contact: dominique.stutzmann@irht.cnrs.fr)"


## 2. Imports

In [2]:
import csv
import glob
import io
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests
from requests.adapters import HTTPAdapter, Retry
from tqdm.auto import tqdm


/pbs/software/redhat-9-x86_64/anaconda/3.12/envs/qiskit-gpu/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Fonctions de lecture des CSV

- `encoding_for_file` choisit l'encodage (UTF-7 ou UTF-8) selon le nom du fichier.
- `sniff_delimiter` détecte le séparateur (tabulation, virgule...).
- `read_csv_robust` lit un fichier en texte avec le bon encodage puis le parse avec pandas.
- `load_all_csvs` charge tous les fichiers du dossier et les concatène, en signalant
  toute incohérence de colonnes entre fichiers.

In [3]:
def encoding_for_file(filepath: str) -> str:
    """Retourne l'encodage à utiliser pour un fichier donné, selon UTF7_FILENAMES."""
    basename = os.path.basename(filepath).lower()
    for pattern in UTF7_FILENAMES:
        if pattern.lower() in basename:
            return "utf-7"
    return DEFAULT_ENCODING

def separator_for_file(filepath: str) -> str | None:
    """Retourne un séparateur forcé si le fichier est dans SEP_OVERRIDES, sinon None (= sniffer)."""
    basename = os.path.basename(filepath).lower()
    for pattern, sep in SEP_OVERRIDES.items():
        if pattern.lower() in basename:
            return sep
    return None

def sniff_delimiter(sample_text: str) -> str:
    """Détecte le séparateur (tabulation, virgule, point-virgule...) à partir d'un échantillon."""
    try:
        dialect = csv.Sniffer().sniff(sample_text, delimiters="\t,;|")
        return dialect.delimiter
    except csv.Error:
        # Par défaut on suppose une tabulation (format observé dans les exemples fournis)
        return "\t"


def read_csv_robust(filepath: str) -> pd.DataFrame:
    encoding = encoding_for_file(filepath)
    with open(filepath, "r", encoding=encoding, errors="strict") as f:
        text = f.read()

    forced_sep = separator_for_file(filepath)
    delimiter = forced_sep if forced_sep is not None else sniff_delimiter(text[:5000])

    df = pd.read_csv(
        io.StringIO(text),
        sep=delimiter,
        dtype=str,
        keep_default_na=False,
        na_values=[""],
    )
    df["__source_file"] = os.path.basename(filepath)
    df["__source_encoding"] = encoding
    df["__source_delimiter"] = delimiter
    return df


def load_all_csvs(input_dir: str, pattern: str) -> pd.DataFrame:
    """Charge tous les CSV du dossier, vérifie la cohérence des colonnes, et les concatène."""
    filepaths = sorted(glob.glob(os.path.join(input_dir, pattern)))
    if not filepaths:
        raise FileNotFoundError(
            f"Aucun fichier trouvé avec le motif '{pattern}' dans '{input_dir}'."
        )

    frames = []
    reference_columns = None
    for fp in filepaths:
        print(f"Lecture: {fp}  (encodage={encoding_for_file(fp)})")
        df = read_csv_robust(fp)
        data_columns = [c for c in df.columns if not c.startswith("__source")]
        if reference_columns is None:
            reference_columns = data_columns
        else:
            missing = set(reference_columns) - set(data_columns)
            extra = set(data_columns) - set(reference_columns)
            if missing or extra:
                print(
                    f"  ATTENTION — colonnes différentes dans {os.path.basename(fp)}: "
                    f"manquantes={missing or None}, en trop={extra or None}"
                )
        frames.append(df)

    combined = pd.concat(frames, ignore_index=True, sort=False)
    print(f"\nTotal: {len(frames)} fichiers, {len(combined)} lignes.")
    return combined


## 4. Fonctions de résolution des redirections HTTP

Pour chaque URL `ark:/...`, on suit les redirections jusqu'à l'URL physique finale.
On utilise `HEAD` (aucun corps de réponse) et, si le serveur ne le supporte pas,
on retombe sur `GET` en streaming (`stream=True`), en fermant la connexion
immédiatement sans jamais lire ni télécharger le contenu de l'image.
Les échecs réseau sont retentés automatiquement (`Retry`) et journalisés en fin
de traitement, sans interrompre le reste du traitement.

In [4]:
def make_session() -> requests.Session:
    session = requests.Session()
    retries = Retry(
        total=MAX_RETRIES,
        backoff_factor=0.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["HEAD", "GET"]),
    )
    adapter = HTTPAdapter(max_retries=retries, pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update({"User-Agent": USER_AGENT})
    return session


'''
def resolve_one_url(url: str, session: requests.Session) -> tuple:
    """
    Résout l'URL finale après redirection(s), sans télécharger le corps de l'image.
    Essaie d'abord HEAD (rapide, pas de corps), puis retombe sur GET en streaming
    (le corps n'est jamais lu ni téléchargé, la connexion est fermée immédiatement).
    Retourne (url_originale, url_resolue, statut, erreur_eventuelle).
    """
    if not url or not isinstance(url, str):
        return (url, None, None, "URL vide ou invalide")

    try:
        resp = session.head(url, allow_redirects=True, timeout=REQUEST_TIMEOUT)
        if resp.status_code >= 400 or resp.status_code == 405:
            raise requests.exceptions.RequestException(
                f"HEAD a échoué ou non supporté (statut {resp.status_code})"
            )
        return (url, resp.url, resp.status_code, None)
    except requests.exceptions.RequestException:
        # Repli sur GET en streaming : on ne lit jamais le corps de la réponse
        try:
            with session.get(url, allow_redirects=True, timeout=REQUEST_TIMEOUT, stream=True) as resp:
                final_url = resp.url
                status = resp.status_code
            return (url, final_url, status, None)
        except requests.exceptions.RequestException as e:
            return (url, None, None, str(e))
        
'''
def resolve_one_url(url: str, session: requests.Session) -> tuple:
    if not url or not isinstance(url, str):
        return (url, None, None, "URL vide ou invalide")

    info_url = to_info_json_url(url)

    try:
        resp = session.head(info_url, allow_redirects=True, timeout=REQUEST_TIMEOUT)
        if resp.status_code >= 400 or resp.status_code == 405:
            raise requests.exceptions.RequestException(
                f"HEAD a échoué ou non supporté (statut {resp.status_code})"
            )
        final_image_url = rebuild_image_url_from_info(resp.url, url)
        return (url, final_image_url, resp.status_code, None)
    except requests.exceptions.RequestException:
        try:
            with session.get(info_url, allow_redirects=True, timeout=REQUEST_TIMEOUT, stream=True) as resp:
                final_image_url = rebuild_image_url_from_info(resp.url, url)
                status = resp.status_code
            return (url, final_image_url, status, None)
        except requests.exceptions.RequestException as e:
            return (url, None, None, str(e))
            

def resolve_urls_parallel(urls: list, preview_n: int = 10) -> dict:
    """Résout une liste d'URL uniques en parallèle. Retourne un dict url -> url_resolue."""
    session = make_session()
    results = {}
    errors = []
    n_previewed = 0

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(resolve_one_url, url, session): url for url in urls}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Résolution des redirections"):
            original_url, resolved_url, status, error = future.result()
            if error:
                errors.append((original_url, error))
                results[original_url] = original_url  # on garde l'URL d'origine en cas d'échec
            else:
                results[original_url] = resolved_url

            if n_previewed < preview_n:
                state = "OK" if not error else f"ÉCHEC ({error})"
                print(f"  [{state}]")
                print(f"    ark  : {original_url}")
                print(f"    final: {results[original_url]}")
                n_previewed += 1

    if errors:
        print(f"\n{len(errors)} URL(s) n'ont pas pu être résolues (URL d'origine conservée) :")
        for url, err in errors[:20]:
            print(f"  - {url} -> {err}")
        if len(errors) > 20:
            print(f"  ... et {len(errors) - 20} autres.")

    return results


def to_info_json_url(image_url: str) -> str:
    """
    Transforme une URL IIIF Image API (.../{region}/{size}/{rotation}/{quality}.{format})
    en URL info.json (.../info.json), qui ne nécessite aucun traitement d'image côté serveur.
    """
    parts = image_url.rstrip("/").split("/")
    base = "/".join(parts[:-4])  # on retire region/size/rotation/quality.format
    return base + "/info.json"


def rebuild_image_url_from_info(resolved_info_url: str, original_image_url: str) -> str:
    """
    À partir de l'URL résolue de info.json et de l'URL image d'origine,
    reconstruit l'URL image finale (même suffixe region/size/rotation/quality
    que l'original, mais avec le host/chemin physique résolu).
    """
    suffix = "/".join(original_image_url.rstrip("/").split("/")[-4:])
    resolved_base = resolved_info_url.rstrip("/")
    if resolved_base.endswith("/info.json"):
        resolved_base = resolved_base[: -len("/info.json")]
    return f"{resolved_base}/{suffix}"

## 5. Chargement et fusion des CSV

In [5]:
def list_columns_per_file(input_dir: str, pattern: str) -> dict:
    """
    Parcourt tous les CSV du dossier et affiche les colonnes de chacun,
    sans charger tout le fichier (juste l'en-tête). Retourne un dict
    {nom_fichier: [colonnes]} pour inspection/comparaison ultérieure.
    """
    filepaths = sorted(glob.glob(os.path.join(input_dir, pattern)))
    if not filepaths:
        raise FileNotFoundError(
            f"Aucun fichier trouvé avec le motif '{pattern}' dans '{input_dir}'."
        )

    columns_by_file = {}
    for fp in filepaths:
        encoding = encoding_for_file(fp)
        with open(fp, "r", encoding=encoding, errors="strict") as f:
            header_line = f.readline()
        forced_sep = separator_for_file(fp)
        delimiter = forced_sep if forced_sep is not None else sniff_delimiter(header_line)

        columns = header_line.rstrip("\r\n").split(delimiter)
        columns_by_file[os.path.basename(fp)] = columns
        print(f"{os.path.basename(fp)}  (encodage={encoding}, séparateur={delimiter!r})")
        print(f"  {len(columns)} colonnes: {columns}")

    # Résumé des écarts par rapport au premier fichier
    reference_name, reference_cols = next(iter(columns_by_file.items()))
    print(f"\nRéférence: {reference_name}")
    for name, cols in columns_by_file.items():
        if cols != reference_cols:
            missing = set(reference_cols) - set(cols)
            extra = set(cols) - set(reference_cols)
            print(f"  ÉCART dans {name}: manquantes={missing or None}, en trop={extra or None}")

    return columns_by_file


columns_by_file = list_columns_per_file(INPUT_DIR, GLOB_PATTERN)

JJ096-JJ099_image_data.csv  (encodage=utf-7, séparateur=',')
  15 colonnes: ['manifestURL', 'canvasId', 'urlImage', 'folderPath', 'imageLabel', 'imageWidthAsDeclared', 'imageHeightAsDeclared', 'htmlCode', 'imageFileName', 'imageWidthAsDownloaded', 'imageHeightAsDownloaded', 'urlResizedImage', 'ResizedImagehtmlCode', 'ResizedImageWidthAsDownloaded', 'ResizedImageHeightAsDownloaded']
JJ100-JJ118_image_data.csv  (encodage=utf-8, séparateur=',')
  15 colonnes: ['manifestURL', 'canvasId', 'urlImage', 'folderPath', 'imageLabel', 'imageWidthAsDeclared', 'imageHeightAsDeclared', 'htmlCode', 'imageFileName', 'imageWidthAsDownloaded', 'imageHeightAsDownloaded', 'urlResizedImage', 'ResizedImagehtmlCode', 'ResizedImageWidthAsDownloaded', 'ResizedImageHeightAsDownloaded']
JJ113_image_data.csv  (encodage=utf-8, séparateur=';')
  18 colonnes: ['manifestURL', 'canvasId', 'imageLabel', 'urlImage', 'htmlCode', 'imageWidthAsDownloaded', 'imageHeightAsDownloaded', 'imageWidthAsDeclared', 'imageHeightAsDec

In [6]:
combined_df = load_all_csvs(INPUT_DIR, GLOB_PATTERN)
combined_df.head()


Lecture: ../List-of-images/JJ096-JJ099_image_data.csv  (encodage=utf-7)
Lecture: ../List-of-images/JJ100-JJ118_image_data.csv  (encodage=utf-8)
Lecture: ../List-of-images/JJ113_image_data.csv  (encodage=utf-8)
  ATTENTION — colonnes différentes dans JJ113_image_data.csv: manquantes={'folderPath'}, en trop={'checksumMD5', 'localExists', 'imageURL_resolved', 'urlResizedImage_resolved'}
Lecture: ../List-of-images/JJ119-JJ132_image_data.csv  (encodage=utf-8)
Lecture: ../List-of-images/JJ133-JJ139_image_data.csv  (encodage=utf-8)
Lecture: ../List-of-images/JJ140-JJ159_image_data.csv  (encodage=utf-8)
Lecture: ../List-of-images/JJ160-JJ169_image_data.csv  (encodage=utf-8)
Lecture: ../List-of-images/JJ170-JJ179_image_data.csv  (encodage=utf-8)
Lecture: ../List-of-images/JJ180-JJ199_image_data.csv  (encodage=utf-8)
Lecture: ../List-of-images/JJ200-JJ211_image_data.csv  (encodage=utf-8)

Total: 10 fichiers, 50256 lignes.


,manifestURL,canvasId,urlImage,folderPath,imageLabel,imageWidthAsDeclared,imageHeightAsDeclared,htmlCode,imageFileName,imageWidthAsDownloaded,...,ResizedImagehtmlCode,ResizedImageWidthAsDownloaded,ResizedImageHeightAsDownloaded,__source_file,__source_encoding,__source_delimiter,imageURL_resolved,urlResizedImage_resolved,localExists,checksumMD5
0,https://api.irht.cnrs.fr/ark:/63955/fvdy3kcmb2...,https://arca.irht.cnrs.fr/iiif/125462/canvas/c...,https://iiif.irht.cnrs.fr/iiif/ark:/63955/vd0q...,images_registres_AN_JJ035_JJ211/images\Paris_A...,plat sup��rieur,4311,6605,200,images_registres_AN_JJ035_JJ211/images\Paris_A...,4311,...,200,1200,1839,JJ096-JJ099_image_data.csv,utf-7,",",NaN,NaN,NaN,NaN
1,https://api.irht.cnrs.fr/ark:/63955/fvdy3kcmb2...,https://arca.irht.cnrs.fr/iiif/125462/canvas/c...,https://iiif.irht.cnrs.fr/iiif/ark:/63955/v6e5...,images_registres_AN_JJ035_JJ211/images\Paris_A...,contre-plat sup��rieur,2048,2048,200,images_registres_AN_JJ035_JJ211/images\Paris_A...,2048,...,200,1200,1200,JJ096-JJ099_image_data.csv,utf-7,",",NaN,NaN,NaN,NaN
2,https://api.irht.cnrs.fr/ark:/63955/fvdy3kcmb2...,https://arca.irht.cnrs.fr/iiif/125462/canvas/c...,https://iiif.irht.cnrs.fr/iiif/ark:/63955/vxyc...,images_registres_AN_JJ035_JJ211/images\Paris_A...,1r,3984,6437,200,images_registres_AN_JJ035_JJ211/images\Paris_A...,3984,...,200,1200,1939,JJ096-JJ099_image_data.csv,utf-7,",",NaN,NaN,NaN,NaN
3,https://api.irht.cnrs.fr/ark:/63955/fvdy3kcmb2...,https://arca.irht.cnrs.fr/iiif/125462/canvas/c...,https://iiif.irht.cnrs.fr/iiif/ark:/63955/vbg5...,images_registres_AN_JJ035_JJ211/images\Paris_A...,1v,3768,6373,200,images_registres_AN_JJ035_JJ211/images\Paris_A...,3768,...,200,1200,2030,JJ096-JJ099_image_data.csv,utf-7,",",NaN,NaN,NaN,NaN
4,https://api.irht.cnrs.fr/ark:/63955/fvdy3kcmb2...,https://arca.irht.cnrs.fr/iiif/125462/canvas/c...,https://iiif.irht.cnrs.fr/iiif/ark:/63955/vr66...,images_registres_AN_JJ035_JJ211/images\Paris_A...,2r,3928,6453,200,images_registres_AN_JJ035_JJ211/images\Paris_A...,3928,...,200,1200,1971,JJ096-JJ099_image_data.csv,utf-7,",",NaN,NaN,NaN,NaN


## 6. Résolution des URL (en parallèle)

Seules les URL **uniques** sont résolues (plusieurs lignes peuvent partager
la même URL `ark`), ce qui évite les requêtes redondantes.

In [7]:
unique_urls = combined_df["urlImage"].dropna().unique().tolist()
print(f"{len(unique_urls)} URL 'ark' uniques à résoudre (sur {len(combined_df)} lignes).")

start = time.time()
resolved_map = resolve_urls_parallel(unique_urls)
print(f"Terminé en {time.time() - start:.1f}s.")

# Aperçu des 10 premiers résultats pour contrôle visuel
print("\nAperçu des résolutions :")
for original_url, resolved_url in list(resolved_map.items())[:10]:
    status = "OK" if resolved_url != original_url else "NON RÉSOLU"
    print(f"  [{status}]")
    print(f"    ark  : {original_url}")
    print(f"    final: {resolved_url}")



49890 URL 'ark' uniques à résoudre (sur 50256 lignes).


Résolution des redirections:   0%|          | 0/49890 [00:00<?, ?it/s]

  [OK]
    ark  : https://iiif.irht.cnrs.fr/iiif/ark:/63955/veumhm82l9ts/full/full/0/default.jpg
    final: https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/Paris/Archives_nationales/FRAN_0021_JJ096/DEPOT/FRAN_0021_0751_A/full/full/0/default.jpg


Résolution des redirections:   0%|          | 2/49890 [00:06<44:29:53,  3.21s/it]

  [OK]
    ark  : https://iiif.irht.cnrs.fr/iiif/ark:/63955/vxyc3rg33kh4/full/full/0/default.jpg
    final: https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/Paris/Archives_nationales/FRAN_0021_JJ096/DEPOT/FRAN_0021_0747_A/full/full/0/default.jpg


...

  [OK]
    ark  : https://iiif.irht.cnrs.fr/iiif/ark:/63955/vf5kcrvmb39y/full/full/0/default.jpg
    final: https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/Paris/Archives_nationales/FRAN_0021_JJ096/DEPOT/FRAN_0021_0752_A/full/full/0/default.jpg
  [OK]
    ark  : https://iiif.irht.cnrs.fr/iiif/ark:/63955/vbetf51dcfuv/full/full/0/default.jpg
    final: https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/Paris/Archives_nationales/FRAN_0021_JJ096/DEPOT/FRAN_0021_0756_A/full/full/0/default.jpg
  [OK]
    ark  : https://iiif.irht.cnrs.fr/iiif/ark:/63955/vk610mhf7aiz/full/full/0/default.jpg
    final: https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/Paris/Archives_nationales/FRAN_0021_JJ096/DEPOT/FRAN_0021_0757_A/full/full/0/default.jpg


Résolution des redirections:   0%|          | 9/49890 [00:18<18:22:36,  1.33s/it]

  [OK]
    ark  : https://iiif.irht.cnrs.fr/iiif/ark:/63955/v4k817bt4q4i/full/full/0/default.jpg
    final: https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/Paris/Archives_nationales/FRAN_0021_JJ096/DEPOT/FRAN_0021_0758_A/full/full/0/default.jpg
  [OK]
    ark  : https://iiif.irht.cnrs.fr/iiif/ark:/63955/vkkomx01rvmy/full/full/0/default.jpg
    final: https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/Paris/Archives_nationales/FRAN_0021_JJ096/DEPOT/FRAN_0021_0759_A/full/full/0/default.jpg


Résolution des redirections: 100%|██████████| 49890/49890 [17:21<00:00, 47.92it/s]  

Terminé en 1042.0s.

Aperçu des résolutions :
  [OK]
    ark  : https://iiif.irht.cnrs.fr/iiif/ark:/63955/veumhm82l9ts/full/full/0/default.jpg
    final: https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/Paris/Archives_nationales/FRAN_0021_JJ096/DEPOT/FRAN_0021_0751_A/full/full/0/default.jpg
  [OK]
    ark  : https://iiif.irht.cnrs.fr/iiif/ark:/63955/vxyc3rg33kh4/full/full/0/default.jpg
    final: https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/Paris/Archives_nationales/FRAN_0021_JJ096/DEPOT/FRAN_0021_0747_A/full/full/0/default.jpg
  [OK]
    ark  : https://iiif.irht.cnrs.fr/iiif/ark:/63955/v6e576jlpbid/full/full/0/default.jpg
    final: https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/Paris/Archives_nationales/FRAN_0021_JJ096/DEPOT/FRAN_0021_0746_AB_A/full/full/0/default.jpg
  [OK]
    ark  : https://iiif.irht.cnrs.fr/iiif/ark:/63955/vd0qz1ihxyni/full/full/0/default.jpg
    final: https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/Paris/Archives_nationales/FRAN_0021_JJ096/DEPOT/FRAN_0021_0746_A/ful

## 7. Construction du fichier final et export

In [9]:
final_df = combined_df.copy()

# Renomme la colonne d'origine (l'URL "ark") et ajoute l'URL physique résolue
final_df = final_df.rename(columns={"urlImage": "urlImage_arkId"})
final_df["urlImage"] = final_df["urlImage_arkId"].map(resolved_map)

# Réordonne les colonnes : on garde l'ordre d'origine, en insérant urlImage_arkId
# à la place de l'ancienne colonne urlImage, suivie de la nouvelle colonne urlImage
original_columns = [c for c in combined_df.columns if not c.startswith("__source")]
new_order = []
for col in original_columns:
    if col == "urlImage":
        new_order += ["urlImage_arkId", "urlImage"]
    else:
        new_order.append(col)

# Colonnes techniques de traçabilité conservées à la fin (utile pour le contrôle qualité)
trace_columns = [c for c in combined_df.columns if c.startswith("__source")]
final_df = final_df[new_order + trace_columns]

# Contrôle qualité rapide
n_unresolved = (final_df["urlImage"] == final_df["urlImage_arkId"]).sum()
print(f"{n_unresolved} lignes où l'URL n'a pas pu être résolue (urlImage == urlImage_arkId).")

final_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
print(f"Fichier écrit: {OUTPUT_FILE} ({len(final_df)} lignes, {len(final_df.columns)} colonnes)")
final_df.head()


0 lignes où l'URL n'a pas pu être résolue (urlImage == urlImage_arkId).
Fichier écrit: ./images_merged_with_resolved_urls.csv (50256 lignes, 23 colonnes)


,manifestURL,canvasId,urlImage_arkId,urlImage,folderPath,imageLabel,imageWidthAsDeclared,imageHeightAsDeclared,htmlCode,imageFileName,...,ResizedImagehtmlCode,ResizedImageWidthAsDownloaded,ResizedImageHeightAsDownloaded,imageURL_resolved,urlResizedImage_resolved,localExists,checksumMD5,__source_file,__source_encoding,__source_delimiter
0,https://api.irht.cnrs.fr/ark:/63955/fvdy3kcmb2...,https://arca.irht.cnrs.fr/iiif/125462/canvas/c...,https://iiif.irht.cnrs.fr/iiif/ark:/63955/vd0q...,https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/...,images_registres_AN_JJ035_JJ211/images\Paris_A...,plat sup��rieur,4311,6605,200,images_registres_AN_JJ035_JJ211/images\Paris_A...,...,200,1200,1839,NaN,NaN,NaN,NaN,JJ096-JJ099_image_data.csv,utf-7,","
1,https://api.irht.cnrs.fr/ark:/63955/fvdy3kcmb2...,https://arca.irht.cnrs.fr/iiif/125462/canvas/c...,https://iiif.irht.cnrs.fr/iiif/ark:/63955/v6e5...,https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/...,images_registres_AN_JJ035_JJ211/images\Paris_A...,contre-plat sup��rieur,2048,2048,200,images_registres_AN_JJ035_JJ211/images\Paris_A...,...,200,1200,1200,NaN,NaN,NaN,NaN,JJ096-JJ099_image_data.csv,utf-7,","
2,https://api.irht.cnrs.fr/ark:/63955/fvdy3kcmb2...,https://arca.irht.cnrs.fr/iiif/125462/canvas/c...,https://iiif.irht.cnrs.fr/iiif/ark:/63955/vxyc...,https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/...,images_registres_AN_JJ035_JJ211/images\Paris_A...,1r,3984,6437,200,images_registres_AN_JJ035_JJ211/images\Paris_A...,...,200,1200,1939,NaN,NaN,NaN,NaN,JJ096-JJ099_image_data.csv,utf-7,","
3,https://api.irht.cnrs.fr/ark:/63955/fvdy3kcmb2...,https://arca.irht.cnrs.fr/iiif/125462/canvas/c...,https://iiif.irht.cnrs.fr/iiif/ark:/63955/vbg5...,https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/...,images_registres_AN_JJ035_JJ211/images\Paris_A...,1v,3768,6373,200,images_registres_AN_JJ035_JJ211/images\Paris_A...,...,200,1200,2030,NaN,NaN,NaN,NaN,JJ096-JJ099_image_data.csv,utf-7,","
4,https://api.irht.cnrs.fr/ark:/63955/fvdy3kcmb2...,https://arca.irht.cnrs.fr/iiif/125462/canvas/c...,https://iiif.irht.cnrs.fr/iiif/ark:/63955/vr66...,https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/...,images_registres_AN_JJ035_JJ211/images\Paris_A...,2r,3928,6453,200,images_registres_AN_JJ035_JJ211/images\Paris_A...,...,200,1200,1971,NaN,NaN,NaN,NaN,JJ096-JJ099_image_data.csv,utf-7,","


## Notes

- Si un fichier autre que `JJ96.csv` doit être lu en UTF-7, ajoutez son nom
  (ou une sous-chaîne de son nom) à `UTF7_FILENAMES` dans la cellule de configuration.
- Si le séparateur détecté automatiquement n'est pas le bon pour un fichier particulier,
  vous pouvez forcer `sep="\t"` (ou `","`) dans `read_csv_robust`.
- Les colonnes `__source_file`, `__source_encoding`, `__source_delimiter` sont ajoutées
  à titre de traçabilité/contrôle qualité en fin de fichier ; supprimez-les avant
  livraison si elles ne sont pas souhaitées.
- Le nombre de requêtes simultanées (`MAX_WORKERS`) peut être réduit si le serveur
  IIIF limite le débit.